# Paper Results Summary (Multi-Sim)

This notebook summarizes `load_cycle_1` results across multiple simulation seeds (`sim1..sim5`) and reports model-level mean/std.

In [35]:

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

BASE_DIR = os.path.abspath('.')
SCENARIOS = [
    'load_cycle_1',
    'load_cycle_2',
    'load_cycle_5',
    'medium_aircraft',
    'large_aircraft',
    'snr_congested',
]
files = []
for scenario in SCENARIOS:
    pattern = os.path.join(BASE_DIR, f"*_observations_{scenario}_sim*.csv")
    files.extend(glob.glob(pattern))
files = sorted(files)
print(f"Found {len(files)} files")
for f in files:
    print(os.path.basename(f))


Found 222 files
BASELINE_observations_large_aircraft_sim1.csv
BASELINE_observations_large_aircraft_sim10.csv
BASELINE_observations_large_aircraft_sim2.csv
BASELINE_observations_large_aircraft_sim3.csv
BASELINE_observations_large_aircraft_sim4.csv
BASELINE_observations_large_aircraft_sim5.csv
BASELINE_observations_large_aircraft_sim6.csv
BASELINE_observations_large_aircraft_sim7.csv
BASELINE_observations_large_aircraft_sim8.csv
BASELINE_observations_large_aircraft_sim9.csv
BASELINE_observations_load_cycle_1_sim1.csv
BASELINE_observations_load_cycle_1_sim10.csv
BASELINE_observations_load_cycle_1_sim2.csv
BASELINE_observations_load_cycle_1_sim3.csv
BASELINE_observations_load_cycle_1_sim4.csv
BASELINE_observations_load_cycle_1_sim5.csv
BASELINE_observations_load_cycle_1_sim6.csv
BASELINE_observations_load_cycle_1_sim7.csv
BASELINE_observations_load_cycle_1_sim8.csv
BASELINE_observations_load_cycle_1_sim9.csv
BASELINE_observations_load_cycle_2_sim1.csv
BASELINE_observations_load_cycle_2_sim

In [36]:

rx = re.compile(r'(?P<model>.+?)_observations_(?P<scenario>' + '|'.join(map(re.escape, SCENARIOS)) + r')_(?P<sim>sim\d+)\.csv$')

runs = []
for fp in files:
    m = rx.match(os.path.basename(fp))
    if not m:
        continue
    model = m.group('model')
    sim = m.group('sim')
    df = pd.read_csv(fp)
    df.columns = [c.strip() for c in df.columns]
    df['scenario'] = m.group('scenario')
    df['model'] = model
    df['sim'] = sim
    runs.append(df)

if not runs:
    raise RuntimeError('No multi-sim CSV files found for the selected scenario.')

all_df = pd.concat(runs, ignore_index=True)
print('Scenarios:', sorted(all_df['scenario'].unique()))
print('Models:', sorted(all_df['model'].unique()))
print('Sims:', sorted(all_df['sim'].unique()))
print('Rows:', len(all_df))
print('Columns:', list(all_df.columns))


Scenarios: ['large_aircraft', 'load_cycle_1', 'load_cycle_2', 'load_cycle_5', 'medium_aircraft', 'snr_congested']
Models: ['BASELINE', 'DQN', 'ODT_FINETUNED', 'PPO']
Sims: ['sim1', 'sim10', 'sim2', 'sim3', 'sim4', 'sim5', 'sim6', 'sim7', 'sim8', 'sim9']
Rows: 240204
Columns: ['step', 'lat', 'lon', 'alt', 'snr', 'load', 'handovers', 'allocated_bw', 'allocation_ratio', 'demand_MB', 'throughput_req', 'queing_delay_s', 'propagation_latency_s', 'transmission_rate_mbps', 'latency_req_s', 'beam_capacity', 'service_drop_s', 'dwell_remaining_s', 'ttt_remaining_s', 'scenario', 'model', 'sim']


In [37]:
all_df['latency_s'] = all_df['queing_delay_s'] + all_df['propagation_latency_s']

## 1) Per-Run Metrics

In [38]:
required = [
    'step','snr','handovers','allocated_bw','allocation_ratio','demand_MB',
    'queing_delay_s','propagation_latency_s','transmission_rate_mbps',
    'service_drop_s','latency_s'
]
missing = [c for c in required if c not in all_df.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

# Normalize sentinels once (global)
all_df = all_df.copy()
all_df['queing_delay_s'] = pd.to_numeric(all_df['queing_delay_s'], errors='coerce')

# Remove any outage/sentinel-like queue delays
all_df.loc[all_df['queing_delay_s'] >= 1, 'queing_delay_s'] = np.nan

# Keep latency consistent with invalid queue rows
all_df.loc[all_df['queing_delay_s'].isna(), 'latency_s'] = np.nan
all_df.loc[all_df['latency_s'] >= 1000, 'latency_s'] = np.nan


def summarize_run(g):
    g_active = g[g['demand_MB'] > 0].copy()

    if g_active.empty:
        alloc_ratio = 1.0
        lat_violation_rate = np.nan
        avg_latency_ms = np.nan
        mean_excess_ms = np.nan
        p95_excess_ms = np.nan
        severity_score = np.nan

        queue_delay_ms_mean = np.nan
        propagation_latency_ms_mean = np.nan
        total_latency_ms_mean = np.nan
    else:
        alloc_ratio = float(g_active['allocation_ratio'].mean())

        lat_req = (
            g_active['latency_req_s']
            if 'latency_req_s' in g_active.columns
            else pd.Series(np.inf, index=g_active.index)
        )

        valid = g_active['latency_s'].notna()
        if valid.any():
            lat_s = g_active.loc[valid, 'latency_s']
            req_s = lat_req.loc[valid]
            excess_s = (lat_s - req_s).clip(lower=0)

            lat_violation_rate = float((excess_s > 0).mean())
            avg_latency_ms = float(lat_s.mean() * 1000.0)
            mean_excess_ms = float(excess_s.mean() * 1000.0)
            p95_excess_ms = float(excess_s.quantile(0.95) * 1000.0)
            severity_score = float((excess_s / req_s).replace([np.inf, -np.inf], np.nan).mean())
        else:
            lat_violation_rate = np.nan
            avg_latency_ms = np.nan
            mean_excess_ms = np.nan
            p95_excess_ms = np.nan
            severity_score = np.nan

        queue_delay_ms_mean = float(g_active['queing_delay_s'].mean() * 1000.0)
        propagation_latency_ms_mean = float(g_active['propagation_latency_s'].mean() * 1000.0)
        total_latency_ms_mean = float(g_active['latency_s'].mean() * 1000.0)

    return pd.Series({
        'steps': g['step'].count(),
        'allocation_ratio_mean': alloc_ratio,
        'allocated_bw_mean': g_active['allocated_bw'].mean() if not g_active.empty else np.nan,
        'demand_mb_mean': g_active['demand_MB'].mean() if not g_active.empty else np.nan,
        'transmission_rate_mean': g_active['transmission_rate_mbps'].mean() if not g_active.empty else np.nan,
        'snr_mean': g_active['snr'].mean() if not g_active.empty else np.nan,

        # latency in ms
        'queue_delay_ms_mean': queue_delay_ms_mean,
        'propagation_latency_ms_mean': propagation_latency_ms_mean,
        'total_latency_ms_mean': total_latency_ms_mean,

        # SLA/violation metrics
        'latency_violation_rate': lat_violation_rate,
        'avg_latency_ms': avg_latency_ms,
        'mean_excess_ms': mean_excess_ms,
        'p95_excess_ms': p95_excess_ms,
        'severity_score': severity_score,

        'service_drop_total': g['service_drop_s'].sum(),
        'service_drop_mean': g['service_drop_s'].mean(),
        'handovers_final': g['handovers'].max(),
    })

per_run = (
    all_df
    .groupby(['model','scenario','sim'], as_index=False)
    .apply(summarize_run)
    .reset_index(drop=True)
)

per_run = per_run.sort_values(['model','scenario','sim']).reset_index(drop=True)
per_run

,model,scenario,sim,steps,allocation_ratio_mean,allocated_bw_mean,demand_mb_mean,transmission_rate_mean,snr_mean,queue_delay_ms_mean,propagation_latency_ms_mean,total_latency_ms_mean,latency_violation_rate,avg_latency_ms,mean_excess_ms,p95_excess_ms,severity_score,service_drop_total,service_drop_mean,handovers_final
0,BASELINE,large_aircraft,sim1,1082.0,0.467229,90.968892,202.078753,145.871774,11.306250,9.142373e+02,54.790907,971.337087,1.000000,971.337087,924.278263,1000.949260,19.534349,20.276242,0.018740,262.0
1,BASELINE,large_aircraft,sim10,1082.0,0.428786,89.513023,221.388296,143.527353,12.144039,6.922535e+02,54.715963,748.852624,1.000000,748.852624,700.241512,752.514156,14.371727,54.477301,0.050349,268.0
2,BASELINE,large_aircraft,sim2,1082.0,0.425571,88.731220,215.608112,142.236441,12.441559,1.913333e+02,54.668958,248.013970,1.000000,248.013970,248.013970,248.013970,NaN,74.010345,0.068401,257.0
3,BASELINE,large_aircraft,sim3,1082.0,0.440957,88.885805,209.555338,142.514531,13.097595,9.336820e+02,54.752581,989.402344,1.000000,989.402344,940.917495,1002.581125,19.357275,145.174491,0.134172,262.0
4,BASELINE,large_aircraft,sim4,1082.0,0.457901,89.186235,204.643626,142.977670,12.356690,4.199599e+02,54.523467,474.774230,1.000000,474.774230,428.477933,986.976336,8.045048,69.652875,0.064374,265.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,PPO,snr_congested,sim5,1082.0,0.982119,33.238505,33.853378,53.301761,17.099733,7.042852e-01,54.599894,55.272251,0.691149,55.272251,5.034382,11.393431,0.090342,97.023185,0.089670,228.0
218,PPO,snr_congested,sim6,1082.0,0.983249,27.050464,27.597245,43.385266,16.707442,1.220209e+00,54.837129,56.048917,0.688617,56.048917,5.891027,11.956870,0.104343,86.507740,0.079952,226.0
219,PPO,snr_congested,sim7,1082.0,0.983639,27.732304,28.275536,44.467416,16.544541,1.965314e+00,54.783888,56.695116,0.757318,56.695116,6.768371,12.394765,0.124351,75.871816,0.070122,225.0
220,PPO,snr_congested,sim8,1082.0,0.982473,33.614917,34.229100,53.897078,16.619798,1.614982e+00,54.799495,56.364225,0.766258,56.364225,6.447865,11.945110,0.117116,76.330935,0.070546,227.0


In [39]:
ODT = per_run[per_run['model'] == 'ODT_FINETUNED']
ODT[['scenario','sim','service_drop_total', 'avg_latency_ms', 'allocation_ratio_mean']]

,scenario,sim,service_drop_total,avg_latency_ms,allocation_ratio_mean
120,large_aircraft,sim1,0.000000,971.337087,0.461011
121,large_aircraft,sim2,0.000000,248.013970,0.434721
122,large_aircraft,sim3,30.561726,989.402344,0.446355
123,large_aircraft,sim4,15.231945,474.774230,0.471120
124,large_aircraft,sim5,0.000000,442.844239,0.468615
125,large_aircraft,sim6,0.000000,424.692558,0.473229
126,large_aircraft,sim9,0.000000,747.830133,0.437195
127,load_cycle_1,sim1,0.000000,56.454737,0.997252
128,load_cycle_1,sim2,0.000000,56.253295,0.995623
129,load_cycle_1,sim3,30.561726,60.910433,0.991558


## 2) Model Mean/Std Across Simulations


metric_cols = [
    'total_latency_mean',
    'allocation_ratio_mean',
    'transmission_rate_mean',
    'service_drop_total',
    'service_drop_mean',
    'handovers_final',
]

summary = (
    per_run
    .groupby(['model', 'scenario'])[metric_cols]
    .agg(['mean','std'])
)

# flatten columns
summary.columns = [f"{m}_{stat}" for m, stat in summary.columns]
summary = summary.reset_index().sort_values('model')
summary


In [44]:
metric_cols = [
    'total_latency_ms_mean',
    'allocation_ratio_mean',
    'service_drop_total',
    'service_drop_mean',
    'transmission_rate_mean',
    'handovers_final',
]

summary = (
    per_run
    .groupby(['scenario', 'model'])[metric_cols]
    .agg(['mean', 'std'])
)

# flatten columns
summary.columns = [f"{m}_{stat}" for m, stat in summary.columns]
summary = summary.reset_index().sort_values(['model', 'scenario'])
summary


,scenario,model,total_latency_ms_mean_mean,total_latency_ms_mean_std,allocation_ratio_mean_mean,allocation_ratio_mean_std,service_drop_total_mean,service_drop_total_std,service_drop_mean_mean,service_drop_mean_std,transmission_rate_mean_mean,transmission_rate_mean_std,handovers_final_mean,handovers_final_std
0,large_aircraft,BASELINE,585.017610,260.962792,0.453488,0.018305,51.881964,53.127836,0.047950,0.049102,144.228387,1.495284,263.000000,3.018462
4,load_cycle_1,BASELINE,56.816860,1.352609,0.985159,0.009877,51.881964,53.127836,0.047950,0.049102,60.286723,12.780918,263.000000,3.018462
8,load_cycle_2,BASELINE,56.084352,1.279864,0.994052,0.003740,10.989630,10.582206,0.010157,0.009780,60.858925,12.829930,263.100000,2.131770
12,load_cycle_5,BASELINE,55.972739,1.158612,0.989134,0.007469,32.286614,28.364894,0.029840,0.026215,60.553711,12.863626,257.900000,1.728840
16,medium_aircraft,BASELINE,293.762463,98.863993,0.731373,0.021800,51.881964,53.127836,0.047950,0.049102,139.814197,2.096715,263.000000,3.018462
20,snr_congested,BASELINE,57.017506,1.568188,0.964549,0.006586,176.273254,37.860879,0.162914,0.034992,59.142161,12.865394,260.000000,3.018462
1,large_aircraft,DQN,584.907805,260.905445,0.455978,0.017256,23.495403,43.317066,0.021715,0.040034,145.137621,1.645830,248.300000,2.710064
5,load_cycle_1,DQN,56.290568,1.840030,0.990952,0.010362,25.973245,47.414929,0.024005,0.043822,60.693318,13.130673,248.900000,2.469818
9,load_cycle_2,DQN,55.877257,1.045719,0.995390,0.003906,6.580282,6.799210,0.006082,0.006284,60.965601,12.868172,246.400000,1.897367
13,load_cycle_5,DQN,55.843341,1.218823,0.991994,0.006233,24.338043,25.698357,0.022494,0.023751,60.751206,12.933317,249.000000,1.699673


In [41]:
import plotly.graph_objects as go
import plotly.io as pio


SCENARIO = "load_cycle_5"
plot_df = all_df[all_df["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    display_map = {"ODT_FINETUNED": "ODT"}
    plot_order = ["BASELINE", "DQN", "PPO", "ODT_FINETUNED"]

    plot_df["throughput_alloc_mbps"] = pd.to_numeric(
        plot_df.get("transmission_rate_mbps", np.nan), errors="coerce"
    )
    plot_df["throughput_req_mbps"] = pd.to_numeric(
        plot_df.get("throughput_req", np.nan), errors="coerce"
    )

# Mean over simulations at each step
mean_df = (
    plot_df.groupby(["model", "step"], as_index=False)
            .agg(
                throughput_alloc_mbps=("throughput_alloc_mbps", "mean"),
                throughput_req_mbps=("throughput_req_mbps", "mean"),
            )
            .sort_values(["model", "step"])
)

mean_df["alloc_smooth"] = (
    mean_df.groupby("model")["throughput_alloc_mbps"]
            .transform(lambda s: s.rolling(15, min_periods=1).mean())
)

# Requested throughput curve from mean across all agents/sims per step
req_df = (
    mean_df.groupby("step", as_index=False)["throughput_req_mbps"]
            .mean()
            .sort_values("step")
)
req_df["req_smooth"] = req_df["throughput_req_mbps"].rolling(15, min_periods=1).mean()

fig = go.Figure()

for agent in plot_order:
    g = mean_df[mean_df["model"] == agent]
    if g.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=g["step"],
            y=g["alloc_smooth"],
            mode="lines",
            name=display_map.get(agent, agent),
        )
    )

fig.add_trace(
    go.Scatter(
        x=req_df["step"],
        y=req_df["req_smooth"],
        mode="lines",
        name="Requested Throughput",
        line=dict(color="black", dash="dash", width=3),
    )
)

fig.update_layout(
    height=520,
    title="Allocated vs Requested Throughput (Load Cycle 3)",
    xaxis_title="Step",
    yaxis_title="Throughput (Mbps)",
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    legend_title="Trace",
)

pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.font.size = 18
pio.templates["plotly_white"].layout.title.font.size = 24
pio.templates["plotly_white"].layout.legend.font.size = 16
pio.templates["plotly_white"].layout.legend.title.font.size = 16
fig.update_layout(
    font=dict(size=18),
    title_font=dict(size=24),
    legend=dict(font=dict(size=16), title=dict(font=dict(size=16))),
    xaxis_title_font=dict(size=20),
    yaxis_title_font=dict(size=20),
    xaxis=dict(tickfont=dict(size=16)),
    yaxis=dict(tickfont=dict(size=16)),
)

fig.show()


In [42]:
import plotly.graph_objects as go

SCENARIO = "medium_aircraft"
plot_df = all_df[all_df["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    display_map = {"ODT_FINETUNED": "ODT"}
    plot_order = ["BASELINE", "DQN", "PPO", "ODT_FINETUNED"]

    plot_df["throughput_alloc_mbps"] = pd.to_numeric(
        plot_df.get("transmission_rate_mbps", np.nan), errors="coerce"
    )
    plot_df["throughput_req_mbps"] = pd.to_numeric(
        plot_df.get("throughput_req", np.nan), errors="coerce"
    )

# Mean over simulations at each step
mean_df = (
    plot_df.groupby(["model", "step"], as_index=False)
            .agg(
                throughput_alloc_mbps=("throughput_alloc_mbps", "mean"),
                throughput_req_mbps=("throughput_req_mbps", "mean"),
            )
            .sort_values(["model", "step"])
)

mean_df["alloc_smooth"] = (
    mean_df.groupby("model")["throughput_alloc_mbps"]
            .transform(lambda s: s.rolling(15, min_periods=1).mean())
)

# Requested throughput curve from mean across all agents/sims per step
req_df = (
    mean_df.groupby("step", as_index=False)["throughput_req_mbps"]
            .mean()
            .sort_values("step")
)
req_df["req_smooth"] = req_df["throughput_req_mbps"].rolling(5, min_periods=1).mean()

fig = go.Figure()

for agent in plot_order:
    g = mean_df[mean_df["model"] == agent]
    if g.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=g["step"],
            y=g["alloc_smooth"],
            mode="lines",
            name=display_map.get(agent, agent),
        )
    )

fig.add_trace(
    go.Scatter(
        x=req_df["step"],
        y=req_df["req_smooth"],
        mode="lines",
        name="Requested Throughput",
        line=dict(color="black", dash="dash", width=3),
    )
)

fig.update_layout(
    height=520,
    title="Allocated vs Requested Throughput (Large Aircraft Scenario)",
    xaxis_title="Step",
    yaxis_title="Throughput (Mbps)",
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    legend_title="Trace",
)

pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.font.size = 18
pio.templates["plotly_white"].layout.title.font.size = 24
pio.templates["plotly_white"].layout.legend.font.size = 16
pio.templates["plotly_white"].layout.legend.title.font.size = 16
fig.update_layout(
    font=dict(size=18),
    title_font=dict(size=24),
    legend=dict(font=dict(size=16), title=dict(font=dict(size=16))),
    xaxis_title_font=dict(size=20),
    yaxis_title_font=dict(size=20),
    xaxis=dict(tickfont=dict(size=16)),
    yaxis=dict(tickfont=dict(size=16)),
)
fig.show()


In [43]:
import pandas as pd
import plotly.graph_objects as go

SCENARIO = "load_cycle_5"
plot_df = all_df[all_df["scenario"] == SCENARIO].copy()

if len(plot_df) == 0:
    print(f"No data found for scenario: {SCENARIO}")
else:
    plot_order = ["BASELINE", "DQN", "PPO", "ODT_FINETUNED"]
    display_map = {"ODT_FINETUNED": "ODT"}
    colors = {
        "BASELINE": "#1f77b4",
        "PPO": "#2ca02c",
        "ODT_FINETUNED": "#d62728",
        "DQN": "#9467bd",
    }

    plot_df["allocated_mbps"] = pd.to_numeric(plot_df["transmission_rate_mbps"], errors="coerce")
    plot_df["requested_mbps"] = pd.to_numeric(plot_df["throughput_req"], errors="coerce")

    mean_df = (
        plot_df.groupby(["model", "step"], as_index=False)
               .agg(
                   allocated_mbps=("allocated_mbps", "mean"),
                   requested_mbps=("requested_mbps", "mean"),
               )
               .sort_values(["model", "step"])
    )

    mean_df["delta_mbps"] = mean_df["requested_mbps"] - mean_df["allocated_mbps"]
    mean_df["delta_smooth"] = (
        mean_df.groupby("model")["delta_mbps"]
               .transform(lambda s: s.rolling(15, min_periods=1).mean())
    )

    fig = go.Figure()

    for agent in plot_order:
        g = mean_df[mean_df["model"] == agent]
        if g.empty:
            continue

        line_color = colors.get(agent, "#444")

        # Area fill to zero
        fig.add_trace(
            go.Scatter(
                x=g["step"],
                y=g["delta_smooth"],
                mode="lines",
                name=display_map.get(agent, agent),
                line=dict(color=line_color, width=2),
                #fill="tozeroy",
                #fillcolor=line_color.replace(")", ",0.18)").replace("rgb", "rgba")
                #if line_color.startswith("rgb") else None,
                #opacity=0.01 ,
            )
        )

    fig.add_hline(y=0, line_dash="dash", line_color="black")

    fig.update_layout(
        height=560,
        title=f"Throughput Delta({SCENARIO})",
        xaxis_title="Step",
        yaxis_title="Delta (Mbps)",
        legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
        legend_title="Method",
    )
    fig.show()
